# D1 · M1.1 — LLM Mechanics for Practitioners
### Annotated Parameter-Comparison Notebook

**Meridian Retail Bank — GenAI Operations Assistant, Phase 0 warm-up**

You're a newly onboarded engineer on Meridian Retail Bank's GenAI Operations team. Before anyone
lets you near a production prompt, you need to understand how the model actually behaves under
different settings — temperature, top-p, max tokens, stop sequences — and how model choice trades
off cost, latency and quality. This notebook is that investigation, done properly and annotated so
someone reviewing your PR can follow your reasoning.

**Core path (required to pass):** Tasks 1–4.
**Stretch path (Diamond tier):** Task 5.

Every code cell that calls the OpenAI API costs a small amount of real money on your API key. The
prompts here are short and the call counts are deliberately kept low — don't loop this notebook
in a `while True`.


## Setup

Your API key comes from the `OPENAI_API_KEY` environment variable — never hard-code it in this
notebook. If you're on the lab VM, this should already be set; if not, ask in the trainer channel
before proceeding, don't paste a key into a cell.


In [ ]:
import os
import time
import json
from openai import OpenAI

assert os.environ.get("OPENAI_API_KEY"), (
    "OPENAI_API_KEY is not set in this environment. "
    "Set it before continuing — never hard-code a key in this notebook."
)

client = OpenAI()
MODEL_PRIMARY = "gpt-4o-mini"   # cheap, fast — used for Tasks 1 and 2
MODEL_COMPARISON = "gpt-4o"     # used alongside MODEL_PRIMARY in Task 3

BANKING_PROMPT = (
    "A Meridian Retail Bank customer writes in: "
    "\"My debit card was declined twice today even though my balance shows sufficient funds. "
    "I need to pay rent this afternoon. What should I do?\" "
    "Draft a short, professional support reply."
)


## Task 1 — Temperature sweep

Call the model at **three temperatures: 0, 0.7, 1.0**, twice each (6 calls total), using the same
`BANKING_PROMPT`. For every call, record: the response text, wall-clock latency in seconds, and
`prompt_tokens` / `completion_tokens` from the response's `usage` field.

Store every run as one dict in the `temperature_runs` list — you'll save this at the end of the
notebook, and the autograder checks its shape, so keep the field names exactly as shown in the
starter stub.


In [ ]:
temperature_runs = []

# TODO: loop over temperatures [0, 0.7, 1.0], 2 calls each.
# For each call:
#   - time the request (time.perf_counter() before/after)
#   - call client.chat.completions.create(model=MODEL_PRIMARY, messages=[...], temperature=t)
#   - append a dict to temperature_runs with keys:
#     "temperature", "latency_seconds", "prompt_tokens", "completion_tokens", "response_text"

# your code here


## Task 2 — Sampling controls: top_p, max_tokens, stop sequences

Three separate checks, same `BANKING_PROMPT`, temperature fixed at `0.7`:

1. **top_p** — call once at `top_p=0.1` and once at `top_p=1.0`.
2. **max_tokens** — call once with `max_tokens=20` and observe the response gets cut off.
3. **stop sequence** — call once with `stop=["."]` and observe the response stops at the first
   period.

Store every run as one dict in `sampling_runs`, same field shape as `temperature_runs` plus a
`variant` label (e.g. `"top_p_0.1"`, `"max_tokens_20"`, `"stop_period"`) so the autograder — and a
human reviewer — can tell which check each entry is.


In [ ]:
sampling_runs = []

# TODO: 3 calls as described above (top_p=0.1, top_p=1.0, max_tokens=20, stop=["."])
# NOTE: that's actually 4 variants across 3 checks (top_p has two calls) — do all 4.
# Append dicts to sampling_runs with keys:
#   "variant", "latency_seconds", "prompt_tokens", "completion_tokens", "response_text"

# your code here


## Task 3 — Model trade-off: cost / latency / quality

Call **both** `MODEL_PRIMARY` and `MODEL_COMPARISON` on the same `BANKING_PROMPT` at `temperature=0.7`.
Record latency and token usage for each, same as before, into `model_comparison_runs`.

Do **not** hard-code a per-token dollar price in this notebook — OpenAI's pricing changes over
time and a stale number here would be exactly the kind of unverified technical claim this
programme avoids shipping. Check the current price on OpenAI's pricing page yourself when you
write your annotation in Task 4, and cite the number as "checked on <date>" rather than treating
it as a fixed fact.


In [ ]:
model_comparison_runs = []

# TODO: call MODEL_PRIMARY and MODEL_COMPARISON on BANKING_PROMPT at temperature=0.7.
# Append dicts to model_comparison_runs with keys:
#   "model", "latency_seconds", "prompt_tokens", "completion_tokens", "response_text"

# your code here


## Task 4 — Annotate your findings

In the markdown cell below, replace the placeholders with your own observations, in your own
words, referencing your actual numbers from Tasks 1–3. This annotation is what makes the notebook
"annotated" — the autograder checks that this cell has been genuinely filled in, not left as the
placeholder text.

Cover at minimum:
- What changed in the *content* of the response as temperature increased from 0 to 1?
- Which sampling control (top_p, max_tokens, or stop) had the most surprising effect, and why?
- Given your latency/token numbers, when would you pick `MODEL_COMPARISON` over `MODEL_PRIMARY`
  for a real Meridian Retail Bank workload — and when would that not be worth it?


### Your annotation

*(replace this placeholder — a real reflection, referencing your actual numbers above)*

- Temperature effect on content: ...
- Most surprising sampling control: ...
- MODEL_PRIMARY vs MODEL_COMPARISON trade-off: ...


In [ ]:
annotation_text = """
REPLACE THIS PLACEHOLDER with your real annotation (see Task 4 above).
"""


## Task 5 — Stretch (Diamond tier): quantify reproducibility

Run `BANKING_PROMPT` **5 times at temperature=0** and **5 times at temperature=1.0** (10 calls
total — mind the cost). Using `difflib.SequenceMatcher`, compute a pairwise similarity ratio
between consecutive responses at each temperature, and store the average similarity for each
group in `reproducibility_variance` as `{"temp_0_avg_similarity": ..., "temp_1_avg_similarity": ...}`.

Then, in one sentence in the cell below, recommend a temperature setting for a banking FAQ
responder (needs to be consistent) versus a marketing copy generator (can be more varied) —
this is exactly the kind of guardrail decision Module M1.3 builds on.


In [ ]:
import difflib

reproducibility_variance = {}

# TODO (stretch): 5 calls at temperature=0, 5 calls at temperature=1.0, same BANKING_PROMPT.
# Compute pairwise SequenceMatcher ratios between consecutive responses at each temperature,
# average them, and store:
# reproducibility_variance = {"temp_0_avg_similarity": ..., "temp_1_avg_similarity": ...}

# your code here (leave reproducibility_variance = {} if skipping the stretch task)


## Render the comparison table

This is the module's actual **Expected Output** per the programme syllabus — a short comparison table of parameter settings versus observed behaviour. Run this before saving.

In [ ]:
# Render the required "short comparison table of parameter settings versus observed
# behaviour" (this is the module's actual Expected Output, not just the saved JSON).
def print_comparison_table(temperature_runs, sampling_runs, model_comparison_runs):
    rows = []
    for r in temperature_runs:
        rows.append((f"temperature={r['temperature']}", r['latency_seconds'], r['prompt_tokens'], r['completion_tokens']))
    for r in sampling_runs:
        rows.append((r['variant'], r['latency_seconds'], r['prompt_tokens'], r['completion_tokens']))
    for r in model_comparison_runs:
        rows.append((r['model'], r['latency_seconds'], r['prompt_tokens'], r['completion_tokens']))

    header = f"{'Setting':<28}{'Latency (s)':<14}{'Prompt tok':<12}{'Completion tok':<15}"
    print(header)
    print("-" * len(header))
    for name, latency, ptok, ctok in rows:
        print(f"{name:<28}{latency:<14}{ptok:<12}{ctok:<15}")

print_comparison_table(temperature_runs, sampling_runs, model_comparison_runs)


## Save your results

Run the cell below last. It assembles everything into `m1_1_comparison_results.json` in this
same folder — the autograder (`Day1/autograder/D1_M1.1_LLM_Mechanics_Autograder.py`) reads this
file, not the notebook itself, so make sure this cell actually runs without error before you
consider the lab done.


In [ ]:
results = {
    "temperature_runs": temperature_runs,
    "sampling_runs": sampling_runs,
    "model_comparison_runs": model_comparison_runs,
    "annotation": annotation_text,
    "reproducibility_variance": reproducibility_variance,  # {} if you skipped the stretch task
}

with open("m1_1_comparison_results.json", "w") as f:
    json.dump(results, f, indent=2)

print("Saved m1_1_comparison_results.json —", len(json.dumps(results)), "bytes")
